# GRM shard processing -- binned phenotype covariance

Uses `GRM-pairs/grm_bin_sharded/grm_shard_tool` (a C++ CLI, not a Python
library -- shelled out to via `subprocess`) to bin the sharded GRM by
relatedness and compute the phenotype cross-product (covariance) within
each bin, for whichever phenotypes you pick.

**Designed to run before all shards are finished.** `grm_shard_tool
accumulate` operates one shard at a time and `merge` just sums whatever
accumulator files exist -- this notebook globs for whatever `.grm.bin.k`
files are actually present under `grm_shard_run.ipynb`'s persist
destination right now, uses only those, and prints how many of the total
it found. Rerunning later, as more shards finish, automatically picks up
more coverage and produces a more complete/precise estimate -- no code
changes needed, just rerun.

Plink-shard-reading only -- doesn't build or run `grm_shard_run.ipynb`/
`grm_shard_batch_submit.ipynb` itself, just consumes whatever shards they've
produced so far.

## Setup

Builds `grm_shard_tool` (idempotent -- `make` no-ops if already built).

In [ ]:
import glob
import os
import re
import subprocess

import pandas as pd
import matplotlib.pyplot as plt

REPO_DIR = os.path.expanduser("~/repos/aou-covariance")  # adjust if your checkout lives elsewhere
TOOL_DIR = f"{REPO_DIR}/GRM-pairs/grm_bin_sharded"
TOOL_BIN = f"{TOOL_DIR}/grm_shard_tool"

subprocess.run(["make"], cwd=TOOL_DIR, check=True)

print("REPO_DIR:", REPO_DIR)
print("TOOL_BIN:", TOOL_BIN, "exists:", os.path.isfile(TOOL_BIN))
assert os.path.isfile(TOOL_BIN), "missing tool binary: " + TOOL_BIN

## Paths

`SHARDS_DIR` is `grm_shard_run.ipynb`'s persist destination -- flat
`grm_shard_{k}_of_{N_SHARDS}.grm.bin.{k}` filenames, one shared
`genome_wide.grm.id`. (Not `grm_shard_batch_submit.ipynb`'s destination --
that one has a `dsub` output-path quirk that puts everything under a
literal, unexpanded `shard_${K}_of_${N_SHARDS}/` folder; point `SHARDS_DIR`
there instead if you're processing Batch-produced shards.)

In [ ]:
N_SHARDS = 300   # must match grm_shard_run.ipynb's N_SHARDS

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
SHARDS_DIR = f"{BUCKET_DIR}/grm_shards"
GRM_ID = f"{SHARDS_DIR}/genome_wide.grm.id"

assert os.path.isfile(GRM_ID), (
    f"missing {GRM_ID} -- run grm_shard_run.ipynb's driver + persist cells first "
    f"(at least one completed shard is needed for its .grm.id)"
)

available_shards = sorted(glob.glob(f"{SHARDS_DIR}/grm_shard_*_of_{N_SHARDS}.grm.bin.*"))
print(f"{len(available_shards)}/{N_SHARDS} shards available so far")
if len(available_shards) < N_SHARDS:
    print("Partial run -- coverage of all pairs is incomplete. Rerun this notebook "
          "later (no changes needed) as more shards finish for a fuller estimate.")

## FID check

`grm_shard_tool`'s pheno lookup keys on the full `(FID, IID)` pair.
`write_grm_pheno()` (`02_phenotype/scripts/local/residualize_lib.R`) sets
`FID = IID = person_id`, but plink's `.grm.id` commonly defaults `FID` to
`"0"` -- confirmed true for this panel (all 177,301 rows). Rather than
trust either convention, `build_aligned_pheno()` below rewrites each
phenotype file's `FID` from the real `.grm.id`, keyed on `IID`, so this
works regardless of which convention is actually in effect.

It also documents exactly which IDs went into each aligned file: a
`#`-prefixed provenance header (source file, generation time, source/
matched/dropped row counts, and a sha256 fingerprint of the sorted
`(FID, IID)` set actually used). `grm_shard_tool` safely ignores these --
it skips any line that doesn't parse to exactly 3 whitespace-separated
tokens, and every header line here has only 2 (the `#` and one
`key=value` token with no internal spaces).</cell id="cell-5">


In [ ]:
import hashlib
from datetime import datetime, timezone

grm_ids = pd.read_csv(GRM_ID, sep=r"\s+", header=None, names=["FID", "IID"], dtype=str)
print(grm_ids.head())
print(f"unique FID values: {grm_ids['FID'].nunique()} across {len(grm_ids)} rows")
id_map = dict(zip(grm_ids["IID"], grm_ids["FID"]))  # real FID per IID, from the GRM itself


def build_aligned_pheno(pheno_path, out_path):
    """Rewrite a residualized phenotype file's FID to match the real .grm.id, keyed on IID.

    Writes a provenance header documenting exactly which IDs went into the result --
    source file, generation time, source/matched/dropped row counts, and a sha256
    fingerprint of the sorted (FID, IID) set actually used (lets you verify two runs,
    or two transforms of the same phenotype, drew on the same underlying ID set without
    storing the full ~177k-row list inline).
    """
    pheno = pd.read_csv(pheno_path, sep=r"\s+", dtype={"FID": str, "IID": str})
    n_source = len(pheno)
    pheno["FID"] = pheno["IID"].map(id_map)
    unmatched = int(pheno["FID"].isna().sum())
    if unmatched:
        pheno = pheno.dropna(subset=["FID"])
    n_matched = len(pheno)

    id_fingerprint = hashlib.sha256(
        "\n".join(f"{fid} {iid}" for fid, iid in sorted(zip(pheno["FID"], pheno["IID"])))
        .encode()
    ).hexdigest()

    with open(out_path, "w") as f:
        # each header line has exactly 2 whitespace-separated tokens ("#" and one
        # "key=value" blob with no internal spaces) -- grm_shard_tool's pheno parser
        # requires 3 tokens to treat a line as data, so these are always safely skipped
        f.write(f"# source={pheno_path}\n")
        f.write(f"# generated={datetime.now(timezone.utc).isoformat()}\n")
        f.write(f"# grm_id={GRM_ID}\n")
        f.write(f"# n_source_rows={n_source}\n")
        f.write(f"# n_matched_grm={n_matched}\n")
        f.write(f"# n_dropped_unmatched={unmatched}\n")
        f.write(f"# id_set_sha256={id_fingerprint}\n")
    pheno.to_csv(out_path, sep=" ", index=False, columns=["FID", "IID", "Y"], mode="a")
    return out_path

## Phenotypes

Systematically discovers every `<phenotype_name>__<transform>__<covariate_set>.pheno`
file in `02_phenotype`'s residualized output -- not a hand-picked list.
`transform`/`covariate_set` are parsed from whatever filenames actually
exist (not hardcoded to the known `invnorm`/`raw` x `base`/`base_pcs`/
`base_pcs_zip3`/`base_pcs_zip3_ses` combination), so this doesn't need
updating if `02_phenotype` adds another covariate set or transform later.</cell id="cell-7">


In [ ]:
PHENO_DIR = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/02_phenotype/residualized"
)

combos = []
for fname in sorted(os.listdir(PHENO_DIR)):
    if not fname.endswith(".pheno"):
        continue
    stem = fname[: -len(".pheno")]
    parts = stem.rsplit("__", 2)   # data-driven split, not hardcoded to known transform/covset names
    if len(parts) != 3:
        print(f"  skipping unparseable filename: {fname}")
        continue
    phenotype_name, transform, covariate_set = parts
    combos.append((phenotype_name, transform, covariate_set, fname))

phenotypes = sorted(set(c[0] for c in combos))
transforms = sorted(set(c[1] for c in combos))
covariate_sets = sorted(set(c[2] for c in combos))
print(f"{len(phenotypes)} phenotypes x up to {len(transforms)} transforms x "
      f"{len(covariate_sets)} covariate sets = {len(combos)} files found")
print(f"phenotypes: {phenotypes}")
print(f"transforms: {transforms}")
print(f"covariate sets: {covariate_sets}")

## Accumulate + merge

One `accumulate` call per available shard, per `(phenotype, transform,
covariate_set)` combo, then one `merge` per combo. `BINS_FILE` reuses
`GRM-pairs/full_grm_bin/bins.txt`'s existing default bin edges rather than
generating new ones. Results are collected as `results[phenotype][transform][covariate_set]`
so the plotting step can group by phenotype/transform and colour by
covariate set. With many phenotypes this runs a lot of small
`accumulate` calls (`len(combos) * len(available_shards)` total) --
each is a lightweight linear scan over one shard's floats, not a
plink-style compute-heavy pass, but it's still worth expecting this
cell to take a while with a large phenotype list.</cell id="cell-9">


In [ ]:
BINS_FILE = f"{REPO_DIR}/GRM-pairs/full_grm_bin/bins.txt"
NBLOCKS = 50
SEED = 1
WORK_DIR = os.path.expanduser("~/grm_pheno_cov")
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isfile(BINS_FILE), "missing bins file: " + BINS_FILE


def run_combo(phenotype_name, transform, covariate_set, pheno_path):
    tag = f"{phenotype_name}__{transform}__{covariate_set}"
    aligned = build_aligned_pheno(pheno_path, f"{WORK_DIR}/{tag}_aligned.pheno")

    acc_files = []
    for shard_bin in available_shards:
        m = re.search(r"grm_shard_(\d+)_of_\d+\.grm\.bin\.\d+$", shard_bin)
        k = int(m.group(1))
        acc_out = f"{WORK_DIR}/{tag}_shard{k}.acc.tsv"
        subprocess.run(
            [TOOL_BIN, "accumulate",
             "--grm-id", GRM_ID, "--shard", shard_bin,
             "--parallel", str(k), str(N_SHARDS),
             "--pheno", aligned, "--bins", BINS_FILE,
             "--nblocks", str(NBLOCKS), "--seed", str(SEED),
             "--out", acc_out],
            check=True,
        )
        acc_files.append(acc_out)

    acc_list = f"{WORK_DIR}/{tag}_acc_list.txt"
    with open(acc_list, "w") as f:
        f.write("\n".join(acc_files) + "\n")

    out_prefix = f"{WORK_DIR}/{tag}_merged"
    subprocess.run(
        [TOOL_BIN, "merge", "--acc-list", acc_list, "--bins", BINS_FILE,
         "--nblocks", str(NBLOCKS), "--out-prefix", out_prefix],
        check=True,
    )
    return pd.read_csv(f"{out_prefix}.full.tsv", sep="\t")


results = {}   # results[phenotype_name][transform][covariate_set] = df
for phenotype_name, transform, covariate_set, fname in combos:
    print(f"[{len(results)+1}] {fname} ...")
    df = run_combo(phenotype_name, transform, covariate_set, f"{PHENO_DIR}/{fname}")
    results.setdefault(phenotype_name, {}).setdefault(transform, {})[covariate_set] = df

print(f"\ndone -- {sum(len(t) for p in results.values() for t in p.values())} combos processed "
      f"across {len(results)} phenotypes")

## Plot -- covariance vs relatedness, one figure per (phenotype, transform)

Classic GREML-style diagnostic: x-axis is the GRM relatedness bin
midpoint, y-axis is the mean phenotype cross-product (covariance) within
that bin, error bars from the delete-block jackknife SE. Two plots per
phenotype (`invnorm` and `raw` kept separate, since they're on different
scales), each with every available covariate set drawn as a differently
coloured line so the effect of adding PCs/zip3/SES on the covariance-vs-
relatedness relationship is visible directly. Titles report how many of
the total shards actually went into this run -- treat as a partial
preview, not a final estimate, until it reads `N_SHARDS/N_SHARDS`. Saved
to `~/grm_pheno_cov/plots/` as well as displayed inline.</cell id="cell-11">


In [ ]:
PLOTS_DIR = os.path.expanduser("~/grm_pheno_cov/plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

cmap = plt.get_cmap("viridis")
covset_color = {
    cs: cmap(i / max(1, len(covariate_sets) - 1)) for i, cs in enumerate(sorted(covariate_sets))
}

saved_paths = []
for phenotype_name in sorted(results):
    for transform in sorted(results[phenotype_name]):
        by_covset = results[phenotype_name][transform]

        fig, ax = plt.subplots(figsize=(8, 6))
        for covariate_set in sorted(by_covset):
            d = by_covset[covariate_set]
            d = d[d["full_n"] > 0]
            ax.errorbar(d["bin_midpoint"], d["full_mean"], yerr=d["jk_se"],
                        label=covariate_set, marker="o", ms=3, lw=1,
                        color=covset_color[covariate_set])
        ax.axvline(0, color="grey", lw=0.5)
        ax.set_xlabel("GRM relatedness (bin midpoint)")
        ax.set_ylabel("Mean phenotype cross-product (covariance)")
        ax.set_title(f"{phenotype_name} ({transform}) -- covariance vs relatedness "
                     f"({len(available_shards)}/{N_SHARDS} shards)")
        ax.legend(title="covariate set")
        plt.tight_layout()

        out_png = f"{PLOTS_DIR}/{phenotype_name}__{transform}.png"
        fig.savefig(out_png, dpi=150)
        saved_paths.append(out_png)
        plt.show()
        plt.close(fig)

print(f"\nsaved {len(saved_paths)} plots to {PLOTS_DIR}")

## Next steps

- Rerun this whole notebook later (no code changes) as more shards finish
  -- `available_shards` picks up whatever's newly present, and the
  `accumulate`/`merge` results get more complete/precise automatically.
- Once `grm_shard_run.ipynb` reports all `N_SHARDS` persisted, a run of
  this notebook is the final, full-coverage result -- worth a final rerun
  even if you've already looked at partial results along the way.
- New phenotype/transform/covariate-set files dropped into `PHENO_DIR`
  are picked up automatically on the next run -- `combos` is discovered
  from what's actually there, nothing to edit.
- Each aligned `.pheno` file's provenance header (`~/grm_pheno_cov/*_aligned.pheno`)
  records exactly which IDs went into it -- check `n_dropped_unmatched`
  there if a phenotype's plot looks off, and compare `id_set_sha256`
  across files to confirm two combos drew on the same underlying sample.